# 現代互動式數據可視化 (2024-2025)

本 Notebook 介紹三個強大的現代互動式可視化庫：

1. **Plotly** - 功能最全面的互動式可視化庫
2. **Altair** - 聲明式可視化，基於 Vega-Lite
3. **hvPlot** - 高級繪圖接口，支持大數據

## 為什麼需要互動式可視化？

### 靜態圖表的限制
- ⚠️ 無法動態探索數據
- ⚠️ 難以展示高維數據
- ⚠️ 缺乏用戶互動
- ⚠️ 不適合儀表板和 Web 應用

### 互動式可視化的優勢
- ✅ 縮放、平移、懸停查看詳情
- ✅ 動態篩選和選擇
- ✅ 易於集成到 Web 應用
- ✅ 更好的數據探索體驗
- ✅ 支持大規模數據集

In [ ]:
# 安裝必要的套件
!pip install plotly altair hvplot bokeh vega_datasets -q

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import altair as alt
import hvplot.pandas
from vega_datasets import data as vega_data

# 設置
alt.data_transformers.enable('default')
alt.renderers.enable('default')

print(f"Plotly version: {px.__version__}")
print(f"Altair version: {alt.__version__}")

## 1. Plotly - 最全面的互動式可視化庫

### 主要特點
- 🎨 豐富的圖表類型（40+ 種）
- 🚀 高性能，支持百萬級數據點
- 📊 支持 3D 可視化
- 🌐 易於導出為 HTML
- 📱 響應式設計，適合移動設備

### 何時使用 Plotly？
- 需要豐富的互動功能
- 構建數據儀表板
- 創建 Web 應用可視化
- 需要 3D 可視化

In [ ]:
# 創建示例數據集
np.random.seed(42)

# 銷售數據
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
df_sales = pd.DataFrame({
    'date': dates,
    'sales': np.random.randn(len(dates)).cumsum() + 100,
    'product': np.random.choice(['A', 'B', 'C', 'D'], len(dates)),
    'region': np.random.choice(['North', 'South', 'East', 'West'], len(dates)),
    'customer_count': np.random.poisson(50, len(dates)),
    'revenue': np.random.gamma(2, 1000, len(dates))
})

df_sales.head()

### 1.1 Plotly Express - 快速創建圖表

In [ ]:
# 基礎折線圖 - 帶互動功能
fig = px.line(
    df_sales, 
    x='date', 
    y='sales',
    color='product',
    title='銷售趨勢（按產品分類）',
    labels={'sales': '銷售額', 'date': '日期', 'product': '產品'},
    hover_data=['customer_count', 'revenue']
)

fig.update_layout(
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

print("💡 提示：")
print("- 懸停查看詳細數據")
print("- 點擊圖例隱藏/顯示系列")
print("- 拖拽縮放區域")
print("- 雙擊重置視圖")

In [ ]:
# 散點圖矩陣 - 探索多變量關係
fig = px.scatter_matrix(
    df_sales.sample(500),
    dimensions=['sales', 'customer_count', 'revenue'],
    color='product',
    title='多變量關係探索',
    opacity=0.6,
    height=700
)

fig.update_traces(diagonal_visible=False)
fig.show()

In [ ]:
# 帶趨勢線的散點圖
fig = px.scatter(
    df_sales,
    x='customer_count',
    y='revenue',
    color='region',
    size='sales',
    trendline='ols',  # 添加回歸線
    title='客戶數量 vs 收入（帶回歸線）',
    labels={'customer_count': '客戶數量', 'revenue': '收入'},
    hover_data=['date', 'product']
)

fig.update_layout(template='plotly_white')
fig.show()

### 1.2 進階圖表類型

In [ ]:
# 箱型圖 - 比較分佈
fig = px.box(
    df_sales,
    x='product',
    y='revenue',
    color='region',
    title='收入分佈（按產品和地區）',
    points='outliers',  # 顯示離群值
    notched=True  # 添加凹口表示置信區間
)

fig.update_layout(
    template='plotly_white',
    height=500
)

fig.show()

In [ ]:
# 小提琴圖 - 更詳細的分佈信息
fig = px.violin(
    df_sales,
    x='product',
    y='revenue',
    color='region',
    box=True,  # 包含箱型圖
    points='all',  # 顯示所有點
    title='收入分佈詳細視圖',
    hover_data=df_sales.columns
)

fig.update_layout(template='plotly_white', height=600)
fig.show()

In [ ]:
# 熱力圖 - 相關性分析
numeric_cols = ['sales', 'customer_count', 'revenue']
corr_matrix = df_sales[numeric_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    title='特徵相關性熱力圖',
    labels={'color': '相關係數'}
)

fig.update_layout(width=600, height=500)
fig.show()

### 1.3 子圖和複雜佈局

In [ ]:
# 創建多個子圖
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '銷售趨勢', 
        '產品分佈', 
        '地區收入', 
        '客戶數量'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'bar'}, {'type': 'histogram'}]
    ]
)

# 子圖 1: 時間序列
for product in df_sales['product'].unique():
    product_data = df_sales[df_sales['product'] == product]
    fig.add_trace(
        go.Scatter(
            x=product_data['date'],
            y=product_data['sales'],
            name=f'Product {product}',
            mode='lines'
        ),
        row=1, col=1
    )

# 子圖 2: 產品銷量
product_sales = df_sales.groupby('product')['sales'].sum()
fig.add_trace(
    go.Bar(x=product_sales.index, y=product_sales.values, name='總銷售額'),
    row=1, col=2
)

# 子圖 3: 地區收入
region_revenue = df_sales.groupby('region')['revenue'].sum()
fig.add_trace(
    go.Bar(x=region_revenue.index, y=region_revenue.values, name='總收入'),
    row=2, col=1
)

# 子圖 4: 客戶數量分佈
fig.add_trace(
    go.Histogram(x=df_sales['customer_count'], name='客戶數量分佈'),
    row=2, col=2
)

fig.update_layout(
    height=800,
    showlegend=True,
    title_text='銷售數據綜合儀表板'
)

fig.show()

### 1.4 動畫和時間序列可視化

In [ ]:
# 準備動畫數據
df_anim = df_sales.copy()
df_anim['year'] = df_anim['date'].dt.year
df_anim['month'] = df_anim['date'].dt.month

monthly_data = df_anim.groupby(['year', 'month', 'product']).agg({
    'sales': 'sum',
    'revenue': 'sum',
    'customer_count': 'sum'
}).reset_index()

monthly_data['year_month'] = pd.to_datetime(
    monthly_data[['year', 'month']].assign(day=1)
)

# 創建動畫散點圖
fig = px.scatter(
    monthly_data,
    x='customer_count',
    y='revenue',
    animation_frame='year_month',
    animation_group='product',
    size='sales',
    color='product',
    hover_name='product',
    title='客戶數量 vs 收入（時間動畫）',
    range_x=[0, monthly_data['customer_count'].max() * 1.1],
    range_y=[0, monthly_data['revenue'].max() * 1.1],
    size_max=50
)

fig.update_layout(
    template='plotly_white',
    height=600
)

fig.show()

print("▶️ 點擊播放按鈕查看動畫！")

## 2. Altair - 聲明式可視化

### 主要特點
- 📝 聲明式語法，簡潔優雅
- 🔗 基於 Vega-Lite 語法
- 🎨 自動處理樣式和佈局
- 🔄 強大的數據轉換能力
- 📊 優秀的統計可視化

### 何時使用 Altair？
- 需要快速探索性分析
- 偏好聲明式編程風格
- 創建複雜的統計圖表
- 需要優雅的代碼

In [ ]:
# 使用 vega_datasets 的示例數據
cars = vega_data.cars()
print(f"Cars dataset shape: {cars.shape}")
cars.head()

### 2.1 基礎圖表

In [ ]:
# 簡潔的散點圖
chart = alt.Chart(cars).mark_circle(size=60).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon', 'Origin']
).properties(
    width=600,
    height=400,
    title='汽車馬力 vs 油耗'
).interactive()

chart

In [ ]:
# 帶回歸線的散點圖
base = alt.Chart(cars).encode(
    x=alt.X('Horsepower:Q', scale=alt.Scale(zero=False)),
    y=alt.Y('Miles_per_Gallon:Q', scale=alt.Scale(zero=False)),
    color='Origin:N'
)

scatter = base.mark_circle(size=60).encode(
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon']
)

# 添加回歸線
regression = base.transform_regression(
    'Horsepower', 'Miles_per_Gallon', groupby=['Origin']
).mark_line()

chart = (scatter + regression).properties(
    width=600,
    height=400,
    title='馬力 vs 油耗（帶回歸線）'
).interactive()

chart

### 2.2 互動式選擇和過濾

In [ ]:
# 互動式區間選擇
brush = alt.selection_interval()

points = alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(brush, 'Origin:N', alt.value('lightgray')),
    size=alt.condition(brush, alt.value(100), alt.value(30)),
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon', 'Origin']
).properties(
    width=600,
    height=400,
    title='互動式數據選擇（拖拽選擇區域）'
).add_params(brush)

# 顯示選中數據的統計
bars = alt.Chart(cars).mark_bar().encode(
    y='Origin:N',
    x='count()',
    color='Origin:N'
).transform_filter(brush).properties(
    width=600,
    title='選中數據統計'
)

chart = points & bars
chart

### 2.3 複雜統計圖表

In [ ]:
# 多視圖儀表板
input_dropdown = alt.binding_select(
    options=['USA', 'Europe', 'Japan'],
    name='選擇地區: '
)
selection = alt.selection_point(
    fields=['Origin'],
    bind=input_dropdown,
    value='USA'
)

# 散點圖
scatter = alt.Chart(cars).mark_circle(size=80).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.Color('Cylinders:O', scale=alt.Scale(scheme='category10')),
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
    tooltip=['Name', 'Horsepower', 'Miles_per_Gallon', 'Cylinders']
).add_params(selection).properties(
    width=300,
    height=300,
    title='馬力 vs 油耗'
)

# 直方圖
hist = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Miles_per_Gallon:Q', bin=alt.Bin(maxbins=20)),
    y='count()',
    color='Origin:N',
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2))
).add_params(selection).properties(
    width=300,
    height=300,
    title='油耗分佈'
)

chart = (scatter | hist)
chart

## 3. hvPlot - 高級繪圖接口

### 主要特點
- 🔌 與 Pandas DataFrame 無縫集成
- 🚀 支持大數據（基於 Bokeh/HoloViews）
- 🎯 簡潔的 API（類似 .plot()）
- 🌊 支持流數據
- 🗺️ 內建地理數據支持

### 何時使用 hvPlot？
- 從 Pandas .plot() 遷移
- 處理大數據集
- 需要地理可視化
- 構建數據管道

In [ ]:
# 使用我們的銷售數據
# hvPlot 使用極其簡潔的語法

# 基礎折線圖
chart = df_sales.hvplot.line(
    x='date',
    y='sales',
    by='product',
    title='銷售趨勢（hvPlot）',
    width=800,
    height=400,
    legend='top_left',
    grid=True
)

chart

In [ ]:
# 散點圖
chart = df_sales.hvplot.scatter(
    x='customer_count',
    y='revenue',
    by='region',
    size='sales',
    title='客戶數量 vs 收入',
    width=700,
    height=500,
    alpha=0.6,
    legend='top_right'
)

chart

In [ ]:
# 箱型圖
chart = df_sales.hvplot.box(
    y='revenue',
    by='product',
    title='收入分佈（按產品）',
    width=600,
    height=400,
    legend=False
)

chart

In [ ]:
# 多圖組合（hvPlot 的強大功能）
# 使用 + 符號組合圖表

line_chart = df_sales.hvplot.line(
    x='date', y='sales', by='product',
    title='銷售趨勢', width=400, height=300
)

scatter_chart = df_sales.hvplot.scatter(
    x='customer_count', y='revenue', by='region',
    title='客戶 vs 收入', width=400, height=300
)

hist_chart = df_sales.hvplot.hist(
    y='sales', by='product', bins=30,
    title='銷售分佈', width=400, height=300, alpha=0.7
)

box_chart = df_sales.hvplot.box(
    y='revenue', by='region',
    title='收入箱型圖', width=400, height=300
)

# 組合成 2x2 布局
layout = (line_chart + scatter_chart + hist_chart + box_chart).cols(2)
layout

## 4. 工具比較與選擇指南

### Plotly vs Altair vs hvPlot

| 特性 | Plotly | Altair | hvPlot |
|------|--------|--------|--------|
| **學習曲線** | 中等 | 低 | 極低 |
| **圖表類型** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **互動性** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **性能** | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **代碼簡潔度** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **定制性** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **3D支持** | ✅ | ❌ | 有限 |
| **動畫** | ✅✅ | ✅ | ✅ |
| **大數據** | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ |
| **儀表板** | ✅✅ (Dash) | ✅ | ✅ (Panel) |

## 5. AI 輔助可視化

### 使用 AI 工具生成可視化代碼

**ChatGPT/Claude 提示詞範例**：

In [ ]:
# 示例：讓 AI 幫你生成複雜可視化

# 提示詞：
prompt = """
我有一個 pandas DataFrame，包含以下列：
- date (日期)
- sales (銷售額)
- product (產品類別：A, B, C, D)
- region (地區：North, South, East, West)
- revenue (收入)

請用 Plotly 創建一個互動式儀表板，包含：
1. 按產品分組的銷售趨勢折線圖
2. 按地區的收入條形圖
3. 銷售額與收入的散點圖（按產品著色）
4. 添加下拉菜單篩選地區

請提供完整代碼並包含註釋。
"""

print("💡 將此提示詞發送給 ChatGPT/Claude，獲取定制化可視化代碼！")
print()
print("其他有用的 AI 輔助場景：")
print("1. 讓 AI 建議最適合你數據的圖表類型")
print("2. 讓 AI 優化現有可視化的美觀度")
print("3. 讓 AI 添加互動功能和動畫")
print("4. 讓 AI 解釋圖表中的模式和異常")

## 6. 實戰技巧和最佳實踐

### 6.1 選擇合適的圖表類型

```python
# 數據關係      →  推薦圖表
# ────────────────────────────
# 趨勢/時間序列  →  折線圖
# 比較類別      →  條形圖
# 分佈         →  直方圖、箱型圖、小提琴圖
# 相關性       →  散點圖、熱力圖
# 組成/占比     →  餅圖、樹狀圖
# 地理         →  地圖
# 高維數據      →  散點圖矩陣、平行坐標
```

### 6.2 性能優化

1. **數據採樣**（大數據集）

In [ ]:
# 對於超過 10,000 行的數據，考慮採樣
if len(df_sales) > 10000:
    sample_df = df_sales.sample(10000)
else:
    sample_df = df_sales

# 或使用聚合
aggregated_df = df_sales.groupby('date').agg({
    'sales': 'mean',
    'revenue': 'sum'
}).reset_index()

2. **使用適當的渲染模式**

In [ ]:
# Plotly: 使用 scattergl 代替 scatter（GPU 加速）
fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=df_sales['customer_count'],
    y=df_sales['revenue'],
    mode='markers'
))
fig.show()

### 6.3 導出和分享

**導出為 HTML**：

In [ ]:
# Plotly
fig = px.line(df_sales, x='date', y='sales', color='product')
fig.write_html('/tmp/sales_chart.html')

print("圖表已保存到 /tmp/sales_chart.html")
print("可以直接在瀏覽器中打開，無需 Python 環境！")

**導出為靜態圖像**：

In [ ]:
# 需要安裝: pip install kaleido
# fig.write_image('/tmp/sales_chart.png', width=1200, height=800)
# fig.write_image('/tmp/sales_chart.pdf')

print("提示：安裝 kaleido 後可以導出 PNG、PDF、SVG 等格式")
print("pip install kaleido")

## 7. 總結與推薦

### 選擇建議

**使用 Plotly 當：**
- ✅ 需要豐富的圖表類型和定制選項
- ✅ 構建專業儀表板和 Web 應用
- ✅ 需要 3D 可視化
- ✅ 團隊熟悉命令式編程

**使用 Altair 當：**
- ✅ 偏好聲明式、優雅的代碼
- ✅ 快速探索性分析
- ✅ 需要複雜的統計變換
- ✅ 數據量適中（< 5,000 行）

**使用 hvPlot 當：**
- ✅ 從 Pandas .plot() 升級
- ✅ 處理大數據集
- ✅ 需要極簡的 API
- ✅ 構建數據管道

### 學習資源

- **Plotly**: [官方文檔](https://plotly.com/python/)
- **Altair**: [官方文檔](https://altair-viz.github.io/)
- **hvPlot**: [官方文檔](https://hvplot.holoviz.org/)
- **示例畫廊**:
  - [Plotly Chart Studio](https://chart-studio.plotly.com/feed/)
  - [Altair Gallery](https://altair-viz.github.io/gallery/)
  - [HoloViz Examples](https://examples.holoviz.org/)

### 下一步

1. 選擇一個庫深入學習
2. 在自己的數據上實踐
3. 探索高級功能（動畫、3D、地圖）
4. 集成到數據分析工作流
5. 學習構建互動式儀表板（Dash、Streamlit、Panel）